# DuckDB & Parquet - Metadata

In [1]:
%config SqlMagic.displaylimit = 100
%config SqlMagic.autopandas = False

In [2]:
%%capture
%load_ext sql

In [3]:
# https://duckdb.org/docs/lts/guides/python/jupyter
%sql duckdb:///:memory: --alias duckdb

Connecting to 'duckdb'

## CSV Generation

In [4]:
from faker import Faker

fake = Faker()

In [5]:
# https://allisonhorst.github.io/palmerpenguins/
SPECIES = ['Adelie', 'Chinstrap', 'Gentoo']
ISLAND = ['Biscoe', 'Dream', 'Torgersen']
SEX = ['female', 'male']

POPULATION_SIZE = 10_000

In [6]:
from csv import DictWriter

with open('../data/penguins.csv', 'w') as file:
    writer = DictWriter(
        file,
        [
            "species",
            "island",
            "bill_length_mm",
            "bill_depth_mm",
            "flipper_length_mm",
            "body_mass_g",
            "sex",
            "year",
            "month",
        ],
        lineterminator="\n",
    )
    writer.writeheader()

    for _ in range(POPULATION_SIZE):
        writer.writerow(
            {
                "species": fake.random_element(SPECIES),
                "island": fake.random_element(ISLAND),
                "bill_length_mm": fake.pyfloat(
                    right_digits=1, min_value=32.1, max_value=59.6
                ),
                "bill_depth_mm": fake.pyfloat(
                    right_digits=1, min_value=13.1, max_value=21.5
                ),
                "flipper_length_mm": fake.random_int(172, 231),
                "body_mass_g": fake.random_int(2700, 6300),
                "sex": fake.random_element(SEX),
                "year": fake.random_int(2020, 2025),
                "month": fake.random_int(1, 12),
            }
        )

In [7]:
%%sql

FROM read_csv('../data/penguins.csv') LIMIT 20;

Running query in 'duckdb'

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,month
Chinstrap,Torgersen,49.3,19.6,175,3410,female,2022,1
Chinstrap,Biscoe,42.8,14.2,183,6082,female,2023,12
Adelie,Biscoe,35.0,18.7,230,4560,female,2024,3
Chinstrap,Biscoe,56.4,13.2,187,4978,male,2022,7
Adelie,Torgersen,45.5,13.7,226,4348,female,2023,2
Chinstrap,Torgersen,36.3,16.6,172,5240,male,2023,6
Chinstrap,Dream,47.7,20.9,186,4537,male,2020,6
Chinstrap,Dream,44.8,19.8,230,4953,male,2024,3
Gentoo,Dream,50.9,17.7,179,4195,female,2023,7
Adelie,Biscoe,32.2,16.8,179,4452,male,2021,2


## Parquet Generation

In [8]:
%%sql

COPY (
    FROM '../data/penguins.csv'
    -- Order by body_mass to organize row groups by the column of interest
    ORDER BY body_mass_g
)
TO '../data/penguins' (
    FORMAT parquet,
    OVERWRITE true,
    -- The Parquet files will be organized by year and month
    PARTITION_BY (year, month),
    -- Some metadata to track the origin of the Parquet file
    KV_METADATA {
        purpose: 'demonstration',
        datasource: 'penguins.csv',
        code: 123_456,
    }
);

Running query in 'duckdb'

Count


Generated folder structure:

In [9]:
%%bash

tree -d -L 2 ../data/penguins

../data/penguins
├── year=2020
│   ├── month=1
│   ├── month=10
│   ├── month=11
│   ├── month=12
│   ├── month=2
│   ├── month=3
│   ├── month=4
│   ├── month=5
│   ├── month=6
│   ├── month=7
│   ├── month=8
│   └── month=9
├── year=2021
│   ├── month=1
│   ├── month=10
│   ├── month=11
│   ├── month=12
│   ├── month=2
│   ├── month=3
│   ├── month=4
│   ├── month=5
│   ├── month=6
│   ├── month=7
│   ├── month=8
│   └── month=9
├── year=2022
│   ├── month=1
│   ├── month=10
│   ├── month=11
│   ├── month=12
│   ├── month=2
│   ├── month=3
│   ├── month=4
│   ├── month=5
│   ├── month=6
│   ├── month=7
│   ├── month=8
│   └── month=9
├── year=2023
│   ├── month=1
│   ├── month=10
│   ├── month=11
│   ├── month=12
│   ├── month=2
│   ├── month=3
│   ├── month=4
│   ├── month=5
│   ├── month=6
│   ├── month=7
│   ├── month=8
│   └── month=9
├── year=2024
│   ├── month=1
│   ├── month=10
│   ├── month=11
│   ├── month=12
│   ├── month=2
│   ├── month=3
│   ├── month=4
│   ├── month=5
│ 

## Parquet Schema & Metadata

**Fields** available in one Parquet file:

In [10]:
%%sql

-- https://duckdb.org/docs/current/data/parquet/metadata#parquet-schema
FROM parquet_schema('../data/penguins/year=2020/month=1/*.parquet');

Running query in 'duckdb'

file_name,name,type,type_length,repetition_type,num_children,converted_type,scale,precision,field_id,logical_type,duckdb_type,column_id
../data/penguins/year=2020/month=1/data_0.parquet,duckdb_schema,None,None,REQUIRED,7,None,None,None,None,None,None,0
../data/penguins/year=2020/month=1/data_0.parquet,species,BYTE_ARRAY,None,OPTIONAL,None,UTF8,None,None,None,None,VARCHAR,1
../data/penguins/year=2020/month=1/data_0.parquet,island,BYTE_ARRAY,None,OPTIONAL,None,UTF8,None,None,None,None,VARCHAR,2
../data/penguins/year=2020/month=1/data_0.parquet,bill_length_mm,DOUBLE,None,OPTIONAL,None,None,None,None,None,None,DOUBLE,3
../data/penguins/year=2020/month=1/data_0.parquet,bill_depth_mm,DOUBLE,None,OPTIONAL,None,None,None,None,None,None,DOUBLE,4
../data/penguins/year=2020/month=1/data_0.parquet,flipper_length_mm,INT64,None,OPTIONAL,None,INT_64,None,None,None,None,BIGINT,5
../data/penguins/year=2020/month=1/data_0.parquet,body_mass_g,INT64,None,OPTIONAL,None,INT_64,None,None,None,None,BIGINT,6
../data/penguins/year=2020/month=1/data_0.parquet,sex,BYTE_ARRAY,None,OPTIONAL,None,UTF8,None,None,None,None,VARCHAR,7


**Metadata** stored in one Parquet file:

In [11]:
%%sql

-- https://duckdb.org/docs/current/data/parquet/metadata#parquet-metadata
FROM parquet_metadata('../data/penguins/year=2020/month=1/*.parquet');

Running query in 'duckdb'

file_name,row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,stats_min,stats_max,stats_null_count,stats_distinct_count,stats_min_value,stats_max_value,compression,encodings,index_page_offset,dictionary_page_offset,data_page_offset,total_compressed_size,total_uncompressed_size,key_value_metadata,bloom_filter_offset,bloom_filter_length,min_is_exact,max_is_exact,row_group_compressed_bytes,geo_bbox,geo_types,encoding_stats,stats_nan_count
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,0,0,158,species,BYTE_ARRAY,Adelie,Gentoo,0,3,Adelie,Gentoo,SNAPPY,PLAIN_DICTIONARY,None,4,52,123,138,{},2701,47,True,True,2697,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,1,0,158,island,BYTE_ARRAY,Biscoe,Torgersen,0,3,Biscoe,Torgersen,SNAPPY,PLAIN_DICTIONARY,None,127,174,122,137,{},2748,47,True,True,2697,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,2,0,158,bill_length_mm,DOUBLE,32.3,58.7,0,None,32.3,58.7,SNAPPY,PLAIN,None,None,249,664,1291,{},None,None,True,True,2697,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",0
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,3,0,158,bill_depth_mm,DOUBLE,13.1,20.9,0,None,13.1,20.9,SNAPPY,PLAIN,None,None,913,562,1291,{},None,None,True,True,2697,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",0
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,4,0,158,flipper_length_mm,INT64,172,231,0,None,172,231,SNAPPY,PLAIN,None,None,1475,456,1291,{},None,None,True,True,2697,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,5,0,158,body_mass_g,INT64,2741,6282,0,None,2741,6282,SNAPPY,PLAIN,None,None,1931,676,1291,{},None,None,True,True,2697,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
../data/penguins/year=2020/month=1/data_0.parquet,0,158,7,5529,6,0,158,sex,BYTE_ARRAY,female,male,0,2,female,male,SNAPPY,PLAIN_DICTIONARY,None,2607,2640,94,90,{},2795,47,True,True,2697,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None


Take a look at the fields `stats_distinct_count`, `stats_min_value` and `stats_max_value`:

In [12]:
%%sql

-- https://duckdb.org/docs/current/data/parquet/metadata#parquet-metadata
SELECT
    path_in_schema,
    type,
    stats_min_value,
    stats_max_value,
    stats_distinct_count,
FROM parquet_metadata('../data/penguins/year=2020/month=1/*.parquet');

Running query in 'duckdb'

path_in_schema,type,stats_min_value,stats_max_value,stats_distinct_count
species,BYTE_ARRAY,Adelie,Gentoo,3
island,BYTE_ARRAY,Biscoe,Torgersen,3
bill_length_mm,DOUBLE,32.3,58.7,None
bill_depth_mm,DOUBLE,13.1,20.9,None
flipper_length_mm,INT64,172,231,None
body_mass_g,INT64,2741,6282,None
sex,BYTE_ARRAY,female,male,2


This metadata will be used at query time for efficient **filtering** and **scanning**.

**Custom Metadata** stored (as text) in one Parquet file:

In [13]:
%%sql

-- https://duckdb.org/docs/current/data/parquet/metadata#parquet-key-value-metadata
FROM parquet_kv_metadata('../data/penguins/year=2020/month=1/*.parquet');

Running query in 'duckdb'

file_name,key,value
../data/penguins/year=2020/month=1/data_0.parquet,b'purpose',b'demonstration'
../data/penguins/year=2020/month=1/data_0.parquet,b'datasource',b'penguins.csv'
../data/penguins/year=2020/month=1/data_0.parquet,b'code',b'123456'


**All Metadata** at once stored in one Parquet file:

In [14]:
%%sql

-- https://duckdb.org/docs/current/data/parquet/metadata#full-metadata
FROM parquet_full_metadata('../data/penguins/year=2020/month=1/*.parquet');

Running query in 'duckdb'

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------